# Sentinel-2 NDVI with GDAL

NDVI = (NIR − Red) / (NIR + Red), read directly from S3 through GDAL's `/vsis3/`, cropped to the area of interest, written as a Cloud-Optimised GeoTIFF. You need `jupyterlab-jstex[s3]`, the GDAL Python bindings (`conda install gdal`, or the image's GDAL), numpy and matplotlib.

In [ ]:
import jstex

jstex.whoami()  # 'anonymous'? Sign in in the widget below or run jstex.login()

In [ ]:
ex = jstex.Explorer()
ex

Choose a Sentinel-2 L2A item in the widget (it becomes `ex.selected_item`), or run the next cell to search from code.

In [ ]:
AOI = [10.0, 45.3, 10.4, 45.6]  # lon/lat bbox (Lake Garda, Italy)
ex.query = {
    **ex.query,
    "collections": ["sentinel-2-l2a"],
    "datetime": {"from": "2024-07-01T00:00:00Z", "to": "2024-07-31T23:59:59Z"},
    "aois": [
        {
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [AOI[0], AOI[1]],
                        [AOI[2], AOI[1]],
                        [AOI[2], AOI[3]],
                        [AOI[0], AOI[3]],
                        [AOI[0], AOI[1]],
                    ]
                ],
            },
            "selected": True,
        }
    ],
    "filters": [{"field": "eo:cloud_cover", "op": "<=", "value": 10}],
}
ex.search(wait=True)
item = ex.selected_item or next(iter(ex.results))
item.id, sorted(item.assets)

In [ ]:
def band(item, common_name):
    """The finest-resolution single-band asset of a band, by its common name (e.g. 'red', 'nir')."""
    found = []
    for key, asset in item.assets.items():
        bands = (
            asset.extra_fields.get("eo:bands") or asset.extra_fields.get("bands") or []
        )
        if any(
            common_name in (b.get("common_name"), b.get("eo:common_name"))
            for b in bands
        ):
            # finest gsd first; single-band assets before composites such as TCI
            found.append((asset.extra_fields.get("gsd") or 999, len(bands), key, asset))
    if not found:
        raise KeyError(f"no {common_name!r} band in {sorted(item.assets)}")
    _, _, key, asset = min(found)
    return key, asset

In [ ]:
import numpy as np
from osgeo import gdal

gdal.UseExceptions()
red_key, red = band(item, "red")
nir_key, nir = band(item, "nir")
for name, value in jstex.s3.gdal_env(red).items():  # S3 key valid up to 8 h
    gdal.SetConfigOption(name, value)


def vsis3(asset):
    loc = jstex.s3.location(asset)
    return f"/vsis3/{loc['Bucket']}/{loc['Key']}"


def scaling(asset):
    """Scale, offset and nodata from the STAC raster metadata (asset level in STAC 1.1, else raster:bands)."""
    ef = asset.extra_fields
    rb = (ef.get("raster:bands") or [{}])[0]
    scale = ef.get("raster:scale", rb.get("scale", 1.0))
    offset = ef.get("raster:offset", rb.get("offset", 0.0))
    return scale, offset, ef.get("nodata", rb.get("nodata"))


def read(asset):
    """Crop to AOI (lon/lat) in the raster's own CRS; reflectance with nodata as NaN."""
    ds = gdal.Warp(
        "", vsis3(asset), format="MEM", outputBounds=AOI, outputBoundsSRS="EPSG:4326"
    )
    arr = ds.GetRasterBand(1).ReadAsArray().astype("float32")
    scale, offset, nodata = scaling(asset)
    refl = arr * scale + offset
    if nodata is not None:
        refl[arr == nodata] = np.nan
    return ds, refl


red_ds, red_r = read(red)
_, nir_r = read(nir)
with np.errstate(divide="ignore", invalid="ignore"):
    ndvi = (nir_r - red_r) / (nir_r + red_r)
print(
    f"NDVI {np.nanmin(ndvi):.2f} … {np.nanmax(ndvi):.2f}, mean {np.nanmean(ndvi):.2f}"
)

In [ ]:
mem = gdal.GetDriverByName("MEM").Create(
    "", red_ds.RasterXSize, red_ds.RasterYSize, 1, gdal.GDT_Float32
)
mem.SetGeoTransform(red_ds.GetGeoTransform())
mem.SetProjection(red_ds.GetProjection())
mem.GetRasterBand(1).WriteArray(np.nan_to_num(ndvi, nan=-9999))
mem.GetRasterBand(1).SetNoDataValue(-9999)
gdal.Translate(
    f"ndvi_{item.id}.tif", mem, format="COG", creationOptions=["COMPRESS=DEFLATE"]
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 6))
plt.imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(label="NDVI")
plt.title(item.id)
plt.axis("off");